# NB06 — Forests and Boosting: Committees of Models
**Machine Learning in Finance · Week 6**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB06_forests_and_boosting.ipynb)

**Finance question:** Can a crowd of simple models beat one smart model?

**Key idea:** A **random forest** is many trees voting, each trained on a random slice of the data. **Gradient boosting** adds trees one at a time, each fixing the last one's mistakes. **Feature importance** shows what a model relies on.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Part A — The crash problem from NB05
We start with the problem where there is something to find: will a stock lose more than half its value
in the next 12 months? Same features, same 12-month gap, same test period (2018 onward).

In [ ]:
CRASH_FEATURES = ["vol_12", "mom_12_1", "size", "beta_12", "max_1", "leverage", "ep", "btm"]
crash_panel = mlfin.add_crash_label(mlfin.make_features(), fall=0.5)
c_train, c_test = mlfin.time_split(crash_panel, "2017-12-31", gap_months=12)
X_train, y_train, _ = mlfin.prepare_xy(c_train, CRASH_FEATURES, "crash_next_12")
X_test, y_test, test_rows = mlfin.prepare_xy(c_test, CRASH_FEATURES, "crash_next_12")
print(f"{len(X_train):,} training rows, {len(X_test):,} test rows, crash rate in test {y_test.mean():.2%}")

## Step 1: one tree vs. a committee of trees
A single deep tree memorizes. A **random forest** grows many deep trees, each on a random sample of rows
and features, and averages their votes: the memorized quirks cancel out, the real patterns add up.

In [ ]:
# 🟡 YOUR TURN
N_TREES = 200    # ← number of trees in the forest: try 10, 50, 500 (more = steadier, slower)
MAX_DEPTH = None # ← None = grow each tree fully; try 4 or 8

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
lone_tree = DecisionTreeClassifier(max_depth=MAX_DEPTH, class_weight="balanced", random_state=0)
forest = RandomForestClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_leaf=5,
                                class_weight="balanced", n_jobs=-1, random_state=0)
mlfin.compare_models({"one deep tree": lone_tree, f"forest of {N_TREES} trees": forest},
                     X_train, y_train, X_test, y_test, dates_test=test_rows["date"])

*What to look for:* both score almost perfectly on the **past** (navy). On the **future** (gold), the lone
tree is close to a coin flip (AUC 0.5) while the forest is much better. Same kind of tree; the
committee makes the difference. "Hit rate in top 10%" = crash rate among the 10% riskiest-rated stocks each month.

## Step 2: boosting — trees that fix each other's mistakes
**Gradient boosting** builds small trees one after another. Each new tree focuses on the cases the
previous ones got wrong. The **learning rate** is how big a correction each tree is allowed to make:
small = careful and slow, big = fast but jumpy (and prone to memorizing).

In [ ]:
# 🟡 YOUR TURN
LEARNING_RATE = 0.05   # ← try 0.01 (careful) and 0.5 (reckless)
N_ROUNDS = 200         # ← how many trees are added, one after another

from sklearn.ensemble import HistGradientBoostingClassifier
boost = HistGradientBoostingClassifier(max_iter=N_ROUNDS, learning_rate=LEARNING_RATE, max_depth=3,
                                       class_weight="balanced", random_state=0)

## Step 3: the full line-up

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")),
    "tree, depth 3": DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0),
    "one deep tree": lone_tree,
    f"random forest ({N_TREES} trees)": forest,
    f"boosting (rate {LEARNING_RATE})": boost,
}
lineup = mlfin.compare_models(models, X_train, y_train, X_test, y_test, dates_test=test_rows["date"])
lineup

*What to look for:* the committees crush the lone deep tree. But do they beat plain **logistic
regression**? Not necessarily: with a handful of sensible features and a simple pattern ("volatile,
indebted stocks crash more"), the simple model is hard to beat. Fancier is not automatically better.

## Step 4: what is the model using? (feature importance)
Every project must show this chart. A **random forest** keeps score of how much each feature helped its
trees split the data. It's quick, but it is measured on the **training** data.

In [ ]:
_ = mlfin.plot_importance(forest, CRASH_FEATURES, title="Random forest: built-in importance (training data)")

**Permutation importance** is the honest version: on the **test** data, shuffle one feature at a time
and see how much worse the model gets. If shuffling a feature doesn't hurt, the model wasn't really
using it. It works for any model, including boosting.

In [ ]:
sample = X_test.sample(8000, random_state=0)              # a sample keeps this quick
_ = mlfin.plot_importance(boost, X=sample, y=y_test.loc[sample.index],
                          title="Boosting: permutation importance on the test period")

*What to look for:* do the two charts agree with each other, and with the logistic weights in NB05
(volatility, leverage)? When different models agree on what matters, trust the story more. When they
disagree, dig in: related features (like volatility and beta) can stand in for each other.

## Part B — Back to beating the index (your project's problem)
Most teams will predict "beat SPY". `mlfin.default_model()` is the course's standard boosting model
(the one the Week 7 backtest uses). How do the committees do on this much harder question?

In [ ]:
# 🟡 YOUR TURN
BEAT_FEATURES = ["mom_12_1", "rev_1", "vol_12", "size", "btm", "ep", "beta_12", "max_1"]   # ← your team's features

beat_panel = mlfin.make_features()
b_train, b_test = mlfin.time_split(beat_panel, "2017-12-31")
Xb_train, yb_train, _ = mlfin.prepare_xy(b_train, BEAT_FEATURES, "beat_next")
Xb_test, yb_test, b_rows = mlfin.prepare_xy(b_test, BEAT_FEATURES, "beat_next")
beat_models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "random forest (100 trees, depth 6)": RandomForestClassifier(n_estimators=100, max_depth=6, n_jobs=-1, random_state=0),
    "course default (boosting)": mlfin.default_model(),
}
beat_lineup = mlfin.compare_models(beat_models, Xb_train, yb_train, Xb_test, yb_test, dates_test=b_rows["date"])
beat_lineup

*What to look for:* every model sits near AUC 0.50 on the future. The top-10% hit rates differ from the
base rate by a percentage point or two at most. Predicting next month's winners is **hard**, whatever
the model. Week 7 asks the money question: can an edge this small survive trading costs?

In [ ]:
sample_b = Xb_test.sample(8000, random_state=0)
_ = mlfin.plot_importance(beat_models["course default (boosting)"], X=sample_b, y=yb_test.loc[sample_b.index],
                          title="Course default model: permutation importance on the test period")

## ⭐ GOING FURTHER (optional)
1. **Regression versions.** Swap to predicting the return itself: `from sklearn.ensemble import
   HistGradientBoostingRegressor`, fit on `prepare_xy(b_train, BEAT_FEATURES, "ret_next")` and score with
   `mlfin.decile_spread(model.predict(X), y, rows["date"])`. Does it beat NB04's Lasso spread?
2. Ask an AI assistant: *"Explain the difference between a random forest and gradient boosting using an
   investment committee analogy."* Is its analogy accurate?
3. Add `"sector"` to `BEAT_FEATURES`. Does any sector show up in the importance chart?

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*